In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [4]:
answer = assistant.rag('How do I run Olama locally?')
print(answer)

I don’t see any FAQ entry in the provided context about running **Ollama** locally.


In [5]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

To run Ollama locally, first install it from [https://ollama.com/download](https://ollama.com/download) for your operating system:

- **macOS**: download and install the `.pkg`
- **Windows**: download and install the `.msi`
- **Linux**: run:
  ```bash
  curl -fsSL https://ollama.com/install.sh | sh
  ```

Then open a terminal and run:

```bash
ollama run llama3
```

This will download the LLaMA 3 model, start it locally, and open a chat-like interface.

To check that the local server is running, you can also run:

```bash
curl http://localhost:11434
```

If you see a response like `{"models": [...]}`, Ollama is running locally.


In [6]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
)

response.output_text

'That depends on the course’s enrollment rules and whether it’s still open.\n\nIf you want, I can help you figure it out. Please share:\n- the course name\n- where it’s offered\n- whether you mean a live class, online course, or university course\n\nIf you’re asking generally: many courses allow late enrollment if there are still spots available, but some have deadlines or prerequisites.'

In [7]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [8]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [9]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [10]:
len(response.output)

1

In [11]:
call = response.output[0]

In [12]:
print(call)

ResponseFunctionToolCall(arguments='{"query":"Can I join the course if I just discovered it? enrollment late join"}', call_id='call_9RAPss1k1iF4luAW3hIBsnUP', name='search', type='function_call', id='fc_090dc64336487837006a2e5f2238c8819cbd010b3687ffcd8c', namespace=None, status='completed')


In [13]:
call.name

'search'

In [14]:
import json

args = json.loads(call.arguments)
args

{'query': 'Can I join the course if I just discovered it? enrollment late join'}

In [15]:
results = search(**args)

In [16]:
result_json = json.dumps(results, indent=2)

In [17]:
function_call_output = {
    "type": "function_call_output",
    'call_id': call.call_id,
    'output': result_json,
}

In [18]:
messages.append(call)

In [19]:
messages.append(function_call_output)

In [20]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"Can I join the course if I just discovered it? enrollment late join"}', call_id='call_9RAPss1k1iF4luAW3hIBsnUP', name='search', type='function_call', id='fc_090dc64336487837006a2e5f2238c8819cbd010b3687ffcd8c', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_9RAPss1k1iF4luAW3hIBsnUP',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "977bf7786c",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Course: I have registered for the LLM Zoomcamp. When can I expect to rec

In [21]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [22]:
print(response.output_text)

Yes — you can still join the course.

If you want a certificate, the key thing is to submit your project while submissions are still open.


In [23]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(619, 33)

In [24]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    # Prices per 1M tokens (example pricing)
    INPUT_PRICE_PER_MILLION = 0.15   # $0.15 / 1M input tokens
    OUTPUT_PRICE_PER_MILLION = 0.60  # $0.60 / 1M output tokens

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION

    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost
    }


# Your tokens
result = calculate_gpt54mini_price(652, 33)

print("Total Cost: $", round(result["total_cost"], 8))

Total Cost: $ 0.0001176


In [25]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        'call_id': call.call_id,
        'output': result_json,
    }

In [26]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'


messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

In [27]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [ ]:
messages.extend(response.output)

for item in response.output:
    if item.type == 'function_call':
        print('function_call:', item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)

    elif item.type == 'message':
        print('ASSISTANT:')        
        print(item.content[0].text)

function_call: search {"query":"join course discovered late enroll course FAQ can I join it"}
function_call: search {"query":"course enrollment late join discovered course FAQ"}
function_call: search {"query":"can I still join the course FAQ discovered course"}


In [ ]:
messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

it = 1

while True:
    print(f'iteration #{it}...')
    has_function_calls = False

    response = openai_client.responses.create(
        model='gpt-5.4-mini',
        input=messages,
        tools=[search_tool]
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == 'function_call':
            print('function_call:', item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == 'message':
            print('ASSISTANT:')
            print(item.content[0].text)
    
    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
function_call: search {"query":"join the course discovered course can I join"}
function_call: search {"query":"enroll in course late join course FAQ"}
function_call: search {"query":"course registration join after start FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, though, you need to submit your project while submissions are still being accepted. Also, if you’re joining now, note that certificates are only available for the live cohort, not self-paced participation.

If you want, I can also help with whether you need to register first, or explain how the certificate/project process works.
break
